# Two-Particle Block-Embedding Matrix Check

This notebook verifies whether the projected block of `TwoParticleSparseBlockEncoding` matches the full benchmark tensor matrix `V_{ij,i'j'}` for a tiny untruncated case.

Current benchmark definition uses center distance in the Coulomb-like factor:
\[R_{(p,r),(q,s)} = d\!\left(\tfrac{r_p+r_r}{2},\tfrac{r_q+r_s}{2}\right).\]

Projection tested:
- sandwich first ancilla `q` in `|0>`
- sandwich `m,l` registers in all-zero state
- compare resulting `(i,j) -> (i',j')` block against full benchmark `V` matrix.


For oracle construction we normalize by `alpha = max|M_oracle|` so entries are in `[0,1]`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import numpy as np
import cirq

from qualtran.cirq_interop import BloqAsCirqGate, get_named_qubits

if (Path.cwd() / 'src').exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / 'src').exists():
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from exciton.benchmark_tensors import generate_v_tensor
from block_encoding.two_particle_row_oracles import build_two_particle_sparse_block_encoding


In [ ]:
# Tiny untruncated setting
D = 1
L_c = 4
R_c = 0 #L_c - 1
R_loc = 0 #L_c - 1
entry_bitsize = 20

# Oracle-form tensor used by the bloq
M_oracle_raw = generate_v_tensor(
    L=L_c, D=D, r_c=R_c, r_loc=R_loc, metric='chebyshev', oracle_convention='direct'
)

# Normalize before building oracle because entry oracle expects values in [0, 1].
alpha = float(np.max(np.abs(M_oracle_raw)))
if alpha <= 0:
    raise ValueError('Degenerate tensor: max|M_oracle_raw| is zero.')
M_oracle = M_oracle_raw / alpha

bloq = build_two_particle_sparse_block_encoding(
    M=M_oracle, D=D, L=L_c, R_c=R_c, R_loc=R_loc, entry_bitsize=entry_bitsize
)

# Full benchmark tensor V_{pqrs} (with center-distance definition), then reshape to matrix V_{ij, i'j'}
V_pqrs = generate_v_tensor(
    L=L_c, D=D, r_c=R_c, r_loc=R_loc, metric='chebyshev', oracle_convention='pqrs'
)
N = L_c ** (2 * D)
V_ref = np.zeros((N, N), dtype=float)
for i in range(L_c):
    for j in range(L_c):
        row = i * L_c + j
        for ip in range(L_c):
            for jp in range(L_c):
                col = ip * L_c + jp
                V_ref[row, col] = V_pqrs[i, j, ip, jp]

# Consistent normalized reference for comparison to the normalized oracle-loaded bloq.
V_ref_norm = V_ref / alpha

print('Parameters:', {'D': D, 'L_c': L_c, 'R_c': R_c, 'R_loc': R_loc, 'entry_bitsize': entry_bitsize})
print('alpha = max|M_oracle_raw|:', alpha)
print('V_ref shape:', V_ref.shape)

print(M_oracle.reshape(L_c**2, (2*R_c+1)*(2*R_loc+1)))
print(V_ref_norm)

Parameters: {'D': 1, 'L_c': 8, 'R_c': 1, 'R_loc': 0, 'entry_bitsize': 20}
alpha = max|M_oracle_raw|: 2.0
V_ref shape: (64, 64)
[[0.         1.         0.13533528]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.13533528 1.         0.13533528]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.13533528 1.         0.13533528]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.     

In [ ]:
# Dense unitary of bloq
named = get_named_qubits(bloq.signature.lefts())
qorder = []
reg_qubits = {}
for reg in bloq.signature.lefts():
    arr = np.array(named[reg.name]).reshape(-1)
    reg_qubits[reg.name] = arr
    qorder.extend(arr.tolist())

op = BloqAsCirqGate(bloq).on(*qorder)
U = cirq.unitary(op)
print('Bloq unitary shape:', U.shape)

In [ ]:
# Projected block B = <0_q,0_m,0_l| U |0_q,0_m,0_l> over i/j subspace
nq = len(qorder)
pos = {q: k for k, q in enumerate(qorder)}

def bits_of(x: int, n: int):
    return [(x >> (n - 1 - t)) & 1 for t in range(n)]  # big-endian

def basis_index(assign: dict[str, int]):
    bits = [0] * nq
    for reg_name, val in assign.items():
        qarr = reg_qubits[reg_name]
        b = bits_of(int(val), len(qarr))
        for qb, bit in zip(qarr, b):
            bits[pos[qb]] = bit
    idx = 0
    for bit in bits:
        idx = (idx << 1) | bit
    return idx

B = np.zeros((N, N), dtype=complex)
for i in range(L_c):
    for j in range(L_c):
        in_col = i * L_c + j
        in_idx = basis_index({'q': 0, 'm': 0, 'l': 0, 'i': i, 'j': j})
        for ip in range(L_c):
            for jp in range(L_c):
                out_row = ip * L_c + jp
                out_idx = basis_index({'q': 0, 'm': 0, 'l': 0, 'i': ip, 'j': jp})
                B[out_row, in_col] = U[out_idx, in_idx]

scale = ((2 * R_c + 1) ** D) * ((2 * R_loc + 1) ** D)
B_scaled = scale * B.real

# Compare against normalized reference consistent with oracle input scaling.
max_imag = float(np.max(np.abs(B.imag)))
max_abs_diff = float(np.max(np.abs(B_scaled - V_ref_norm)))

# Best scalar fit s minimizing ||B_scaled - s V_ref_norm||_F
num = float(np.vdot(V_ref_norm, B_scaled).real)
den = float(np.vdot(V_ref_norm, V_ref_norm).real)
s = num / den if den > 0 else 0.0
max_abs_diff_scaled = float(np.max(np.abs(B_scaled - s * V_ref_norm)))

print('scale factor (2R_c+1)^D (2R_loc+1)^D:', scale)
print('oracle normalization alpha:', alpha)
print('max imag in B:', max_imag)
print('max |(scale*B) - V_ref_norm|:', max_abs_diff)
print('best scalar s in (scale*B) ~ s*V_ref_norm:', s)
print('max |(scale*B) - s V_ref_norm|:', max_abs_diff_scaled)

print('\n(scale*B) (real):')
print(np.array2string(B_scaled, precision=6, suppress_small=True))
print('\nV_ref_norm:')
print(np.array2string(V_ref_norm, precision=6, suppress_small=True))


scale factor (2R_c+1)^D (2R_loc+1)^D: 3
oracle normalization alpha: 2.0
max imag in B: 3.75629373905307e-18
max |(scale*B) - V_ref_norm|: 5.385686646097732e-07
best scalar s in (scale*B) ~ s*V_ref_norm: 0.9999998935923702
max |(scale*B) - s V_ref_norm|: 5.241679578993352e-07

(scale*B) (real):
[[1.       0.       0.       0.       0.       0.135335 0.       0.
  0.       0.       0.       0.       0.       0.       0.       0.      ]
 [0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.       0.       0.       0.      ]
 [0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.       0.       0.       0.      ]
 [0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.       0.       0.       0.      ]
 [0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.       0.       0.       

## Interpretation
Because the oracle input is normalized by `alpha = max|M_oracle_raw|`, the appropriate target is `V_ref_norm = V_ref / alpha`. This check compares `((2R_c+1)^D (2R_{\mathrm{loc}}+1)^D) B` against `V_ref_norm`.
